# ReceiptVLM — CUDA latency & memory benchmarkMeasures per-receipt latency and peak GPU memory for the re-trained adapter on a CUDA GPU,in **fp16** and **NF4**, back to back in one session so the two share identical hardwareand inputs.## Why the sample is what it is`zeroshot.load_image_ids(split, limit)` takes `records[:limit]` from `test.jsonl` in fileorder, so every subset in this project is a **prefix** of the same list. The first 30 aretherefore a strict subset of the first 60, and one run over 60 receipts is directlycomparable to both existing measurements:| existing measurement | n | source ||---|---|---|| MLX FP16 17.81 s, 8.78 GB peak | 60 | `src/quantize.py` || MLX INT8 12.0 s, 5.4 GB · INT4 9.5 s, 4.4 GB | 60 | `src/quantize.py` || CUDA T4 NF4 24.1 s | 30 | the accuracy gate || MPS fp16 12.5 s mean / 11.1 median, 7.6 GB | 30 | measured locally |This notebook reports aggregates over **both** the full 60 and the first 30, so no row inthat table needs re-running to compare.## Method notes* A **warm-up receipt runs before timing starts** and is excluded. The first generate on a  fresh CUDA context pays for kernel autotuning and lazy module init, which would  otherwise inflate the first measurement by several seconds.* Peak memory is read from `torch.cuda.max_memory_allocated()` (tensor peak) and  `max_memory_reserved()` (caching-allocator footprint), reset per precision.  **`max_memory_allocated` is the right column to compare against the MLX rows** --  `quantize.py` reports `mx.get_peak_memory()`, which is likewise an allocator tensor  peak. `max_memory_reserved` includes pool overhead torch never handed to a tensor, so it  reads high against MLX.* Generated character count is recorded per receipt, because latency here is dominated by  output length rather than image size — a 50-line-item receipt takes ~7x a 2-item one.## InputsAttach the training run's **Notebook Output** (for `final_peft/`) and the`receiptvlm-data` Dataset (for `test.jsonl`; add `eval.py` + siblings if you also wantmicro-F1). Use **GPU T4 x2** and **Internet on**.

In [ ]:
import torchopen("constraints.txt", "w").write(f"torch=={torch.__version__.split('+')[0]}\n")

In [ ]:
!pip install -q -U -c constraints.txt "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" "bitsandbytes>=0.44" "safetensors>=0.4"

## Configuration

In [ ]:
import gc, json, statistics, sys, timefrom pathlib import Pathimport torch# 60 covers quantize.py's subset; the first 30 of it match the accuracy gate and the MPS# run. Lower this if you are short on GPU quota -- 30 still matches two of the four rows.N_RECEIPTS = 60PRECISIONS = ["fp16", "nf4"]     # measured back to back on the same hardwareMAX_NEW_TOKENS = 1536IMAGE_RESIZE = (768, 1024)BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"EXPECT_MODULES = 252PROMPT = ("Extract the receipt fields as JSON with keys store, date, tax, tip, "          "subtotal, total, line_items (each {name, price}). Use null for missing "          "scalar fields and [] for no line items.")OUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("out")OUT_ROOT.mkdir(parents=True, exist_ok=True)if not torch.cuda.is_available():    raise SystemExit("No GPU. Set Accelerator -> GPU T4 x2.")major, minor = torch.cuda.get_device_capability(0)sm = f"sm_{major}{minor}"if sm not in torch.cuda.get_arch_list():    raise SystemExit(f"torch has no kernels for {torch.cuda.get_device_name(0)} ({sm}); "                     f"it targets {torch.cuda.get_arch_list()}. Use 'GPU T4 x2'.")if "nf4" in PRECISIONS and (major, minor) < (7, 5):    raise SystemExit(f"bitsandbytes 4-bit needs sm_75+; this is {sm}.")DEVICE_NAME = torch.cuda.get_device_name(0)TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9print(f"{DEVICE_NAME} ({sm})  {TOTAL_VRAM:.1f} GB VRAM")print(f"benchmarking {PRECISIONS} over {N_RECEIPTS} receipts")

## Locate inputs

In [ ]:
INPUT_ROOT = Path("/kaggle/input")def find_input(name, must_contain=None):    def ok(p):        return (p / must_contain).exists() if must_contain else p.exists()    if INPUT_ROOT.exists():        for p in sorted(INPUT_ROOT.rglob(name)):            if ok(p):                return p    for p in (Path("data/processed")/name, Path("src")/name,              Path("checkpoints")/name, Path(name)):        if p.exists() and ok(p):            return p    return None# The training output holds step_* checkpoints as well as final_peft, and "checkpoints/"# sorts before "final_peft/" -- prefer the final adapter explicitly._cfgs = sorted(INPUT_ROOT.rglob("adapter_config.json")) if INPUT_ROOT.exists() else []if not _cfgs and Path("checkpoints/final_peft/adapter_config.json").exists():    _cfgs = [Path("checkpoints/final_peft/adapter_config.json")]adapter_cfg = next((p for p in _cfgs if "final_peft" in str(p)), _cfgs[0] if _cfgs else None)adapter_dir = adapter_cfg.parent if adapter_cfg else Nonetest_jsonl = find_input("test.jsonl")eval_py = find_input("eval.py")            # optional: enables micro-F1 alongside timingimg_root = find_input("wildreceipt", must_contain="image_files")if img_root is None:    dest = OUT_ROOT / "wildreceipt"    dest.mkdir(parents=True, exist_ok=True)    print("downloading WildReceipt (~179 MB)...")    !curl -sL -o {OUT_ROOT}/wildreceipt.tar https://download.openmmlab.com/mmocr/data/wildreceipt.tar    !tar -xf {OUT_ROOT}/wildreceipt.tar -C {dest} --strip-components=1    img_root = destif adapter_dir is None or test_jsonl is None:    raise SystemExit(f"Missing inputs. adapter={adapter_dir} test.jsonl={test_jsonl}")SCORE = eval_py is not None and all((eval_py.parent/s).exists()                                    for s in ["repair.py", "schema.py", "zeroshot.py"])print("adapter :", adapter_dir)print("test set:", test_jsonl)print("images  :", img_root)print("micro-F1:", "yes" if SCORE else "no (eval.py + siblings not attached)")gold = {json.loads(l)["image_id"]: json.loads(l) for l in test_jsonl.open() if l.strip()}# Same prefix ordering as zeroshot.load_image_ids, so subsets nest.IDS = list(gold)[:N_RECEIPTS]missing = [i for i in IDS if not (img_root / i).exists()]assert not missing, f"{len(missing)} images missing, e.g. {missing[0]}"print(f"\n{len(IDS)} receipts; line-item counts "      f"min {min(len(gold[i].get('line_items') or []) for i in IDS)} / "      f"max {max(len(gold[i].get('line_items') or []) for i in IDS)}")

## Benchmark

In [ ]:
from PIL import Imagefrom transformers import AutoProcessor, Qwen2_5_VLForConditionalGenerationfrom peft import PeftModelprocessor = AutoProcessor.from_pretrained(BASE_MODEL)def fit_within(img, max_w, max_h):    '''Port of mlx_vlm.utils.resize_image: fit the box, aspect preserved, no clamp.'''    ratio = min(max_w / img.width, max_h / img.height)    return img.resize((int(img.width * ratio), int(img.height * ratio)))def load_model(precision):    kwargs = {"dtype": torch.float16, "attn_implementation": "sdpa"}    if precision == "nf4":        from transformers import BitsAndBytesConfig        kwargs["quantization_config"] = BitsAndBytesConfig(            load_in_4bit=True, bnb_4bit_quant_type="nf4",            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE_MODEL, **kwargs)    m = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=False)    m = (m.eval() if precision == "nf4" else m.to("cuda").eval())    m.config.use_cache = True    n = sum(1 for _, mod in m.named_modules()            if hasattr(getattr(mod, "lora_A", None), "keys"))    assert n == EXPECT_MODULES, f"expected {EXPECT_MODULES} lora modules, got {n}"    return mdef generate(m, image_path):    img = fit_within(Image.open(image_path).convert("RGB"), *IMAGE_RESIZE)    messages = [{"role": "user", "content": [{"type": "image"},                                             {"type": "text", "text": PROMPT}]}]    text = processor.apply_chat_template(messages, tokenize=False,                                         add_generation_prompt=True)    enc = processor(text=[text], images=[img], return_tensors="pt").to("cuda")    with torch.inference_mode():        out = m.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)    n_new = out.shape[1] - enc["input_ids"].shape[1]    return processor.batch_decode(out[:, enc["input_ids"].shape[1]:],                                  skip_special_tokens=True)[0], n_new

In [ ]:
results = {}for precision in PRECISIONS:    print(f"\n{'=' * 66}\n{precision.upper()}\n{'=' * 66}")    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()    t0 = time.time()    model = load_model(precision)    load_s = time.time() - t0    weights_gb = torch.cuda.memory_allocated() / 1e9    print(f"load {load_s:.1f}s | weights resident {weights_gb:.2f} GB")    # Warm-up: the first generate pays for kernel autotuning and lazy init. Excluded.    t = time.time(); generate(model, img_root / IDS[0]); warm_s = time.time() - t    print(f"warm-up (excluded): {warm_s:.1f}s")    torch.cuda.reset_peak_memory_stats()    lat, chars, toks, preds = [], [], [], {}    t_all = time.time()    for i, iid in enumerate(IDS, 1):        t = time.time()        raw, n_new = generate(model, img_root / iid)        lat.append(time.time() - t); chars.append(len(raw)); toks.append(n_new)        preds[iid] = raw        if i % 10 == 0:            print(f"  {i}/{len(IDS)}  mean {statistics.mean(lat):.1f}s", flush=True)    total_s = time.time() - t_all    peak_alloc = torch.cuda.max_memory_allocated() / 1e9    peak_resv = torch.cuda.max_memory_reserved() / 1e9    def agg(v):        return {"mean": statistics.mean(v), "median": statistics.median(v),                "min": min(v), "max": max(v),                "p95": sorted(v)[int(0.95 * (len(v) - 1))]}    results[precision] = {        "device": DEVICE_NAME, "n": len(IDS), "load_s": load_s,        "warmup_s": warm_s, "total_s": total_s,        "weights_gb": weights_gb, "peak_alloc_gb": peak_alloc, "peak_reserved_gb": peak_resv,        "latency_all": agg(lat), "latency_first30": agg(lat[:30]),        "tokens": agg(toks), "chars": agg(chars),        "tokens_per_s": sum(toks) / sum(lat),        "_lat": lat, "_toks": toks, "_preds": preds,    }    print(f"\n{precision}: {statistics.mean(lat):.1f}s/receipt over {len(IDS)}  "          f"| peak alloc {peak_alloc:.2f} GB, reserved {peak_resv:.2f} GB")    del model; gc.collect(); torch.cuda.empty_cache()print("\ndone")

## Results

In [ ]:
print(f"Device: {DEVICE_NAME}  ({TOTAL_VRAM:.1f} GB VRAM)\n")hdr = f"{'precision':<10}{'n':>4}{'mean':>9}{'median':>9}{'p95':>9}{'max':>9}{'tok/s':>8}{'peak GB*':>10}"print(hdr); print("-" * len(hdr))# * peak allocated -- the column comparable to quantize.py's MLX numbersfor p, r in results.items():    a = r["latency_all"]    print(f"{p:<10}{r['n']:>4}{a['mean']:>8.1f}s{a['median']:>8.1f}s{a['p95']:>8.1f}s"          f"{a['max']:>8.1f}s{r['tokens_per_s']:>8.1f}{r['peak_alloc_gb']:>10.2f}")print(f"\nFirst 30 only (matches the accuracy gate and the MPS run):")for p, r in results.items():    a = r["latency_first30"]    print(f"  {p:<8} mean {a['mean']:.1f}s  median {a['median']:.1f}s  max {a['max']:.1f}s")print(f"\nMemory breakdown:")for p, r in results.items():    print(f"  {p:<8} weights {r['weights_gb']:.2f} GB | peak allocated "          f"{r['peak_alloc_gb']:.2f} GB | peak reserved {r['peak_reserved_gb']:.2f} GB "          f"| load {r['load_s']:.1f}s")print("\nReference points (different hardware, same receipt prefix):")print(f"  {'MLX INT4  (n=60)':<24} 9.5s   4.4 GB")print(f"  {'MLX INT8  (n=60)':<24} 12.0s  5.4 GB")print(f"  {'MLX FP16  (n=60)':<24} 17.8s  8.8 GB")print(f"  {'MPS fp16  (n=30)':<24} 12.5s  7.6 GB allocated")print(f"  {'CUDA NF4  (n=30, gate)':<24} 24.1s  -")payload = {p: {k: v for k, v in r.items() if not k.startswith('_')}           for p, r in results.items()}(OUT_ROOT / "cuda_benchmark.json").write_text(json.dumps(payload, indent=2))print(f"\nwrote {OUT_ROOT/'cuda_benchmark.json'}")

### Latency is driven by output lengthPer-receipt time tracks generated tokens far more than image size, which is why a50-line-item receipt costs several times a 2-item one. The correlation below should bestrongly positive; if it is not, something other than decoding dominates.

In [ ]:
for p, r in results.items():    lat, toks = r["_lat"], r["_toks"]    n = len(lat)    mt, ml = statistics.mean(toks), statistics.mean(lat)    cov = sum((toks[i]-mt)*(lat[i]-ml) for i in range(n)) / n    sd = (statistics.pstdev(toks) * statistics.pstdev(lat)) or 1    print(f"{p}: corr(tokens, latency) = {cov/sd:.3f}  |  "          f"tokens {min(toks)}-{max(toks)}  latency {min(lat):.1f}-{max(lat):.1f}s")    order = sorted(range(n), key=lambda i: toks[i])    print(f"   fastest: {toks[order[0]]:>4} tok -> {lat[order[0]]:.1f}s"          f"   slowest: {toks[order[-1]]:>4} tok -> {lat[order[-1]]:.1f}s")

## Optional: micro-F1 on the same run

In [ ]:
if not SCORE:    print("Skipped -- attach eval.py, repair.py, schema.py and zeroshot.py "          "(same folder) to also get accuracy for these receipts.")else:    sys.path.insert(0, str(eval_py.parent))    from eval import bootstrap_micro_f1, evaluate    from repair import repair_json    from zeroshot import normalize    for p, r in results.items():        preds = {}        for iid, raw in r["_preds"].items():            parsed, _ = repair_json(raw)            preds[iid] = {"image_id": iid, **normalize(parsed)}        g = {i: gold[i] for i in preds}        _, micro, n_scored = evaluate(g, preds)        lo, hi = bootstrap_micro_f1(g, preds, n=1000)        print(f"{p:<6} micro-F1 {micro[2]:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  (n={n_scored})")        sub = {i: preds[i] for i in list(preds)[:30]}        _, micro30, _ = evaluate({i: gold[i] for i in sub}, sub)        print(f"       first 30: {micro30[2]:.3f}   (gate reported fp16 0.760 / NF4 0.722)")

## What to send backPaste the **Results** cell output, or attach `cuda_benchmark.json` from the Output tab.That completes the latency/memory table across all four backends.Two caveats that will apply to whatever it prints:* A **T4 is not a ZeroGPU GPU.** ZeroGPU allocates a far newer accelerator, so treat the  T4 numbers as a lower bound for the hosted Space, not a prediction of it.* NF4 trades speed for memory. If NF4 is slower than fp16 here while using less VRAM,  that is expected — dequantization costs time on every matmul — and it reinforces serving  fp16, which already scored higher (0.760 vs 0.722).